In [2]:
import os
import numpy as np
import cv2
from PIL import Image, ImageStat
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.preprocessing import StandardScaler

def estimate_jpeg_quality(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, laplacian = cv2.threshold(cv2.convertScaleAbs(cv2.Laplacian(gray, 3)), 0, 255, cv2.THRESH_BINARY)
    return np.mean(laplacian)

def extract_features(image_path):
    try:
        img = Image.open(image_path)
        img_cv = cv2.imread(image_path)
        if img_cv is None:
            print(f"Warning: Unable to read {image_path} with OpenCV. Skipping...")
            return None
    except:
        print(f"Error: Unable to open {image_path}. Skipping...")
        return None
    
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    features = []
    
    # 1. Estimated JPEG Quality
    jpeg_quality = estimate_jpeg_quality(img_cv)
    features.append(jpeg_quality)
    
    # 2. Image sharpness (using variance of Laplacian)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    features.append(sharpness)
    
    # 3. RGB channel statistics
    stat = ImageStat.Stat(img)
    for channel in range(3):  # R, G, B
        features.extend([stat.mean[channel], stat.rms[channel], stat.var[channel]])
    
    # 4. Image entropy (measure of image complexity)
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    hist = hist / hist.sum()
    entropy = -np.sum(hist * np.log2(hist + 1e-10))
    features.append(entropy)
    
    # 5. Edge density (using Sobel operator)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    edge_density = np.mean(np.sqrt(sobel_x**2 + sobel_y**2))
    features.append(edge_density)
    
    # 6. Color Range (max - min) for each channel
    for i in range(3):
        channel = img_cv[:,:,i]
        features.append(np.max(channel) - np.min(channel))
    
    return features

# Paths to your dataset
real_path = "C://Users//Sinchan A//Desktop//Internship//vid//real"
fake_path = "C://Users//Sinchan A//Desktop//Internship//vid//fake"

# Lists to store features and labels
X = []
y = []

# Function to safely add features
def add_features(img_path, label):
    features = extract_features(img_path)
    if features is not None:
        X.append(features)
        y.append(label)

# Extract features
for img_name in os.listdir(real_path):
    img_path = os.path.join(real_path, img_name)
    add_features(img_path, 0)  # 0 for real

for img_name in os.listdir(fake_path):
    img_path = os.path.join(fake_path, img_name)
    add_features(img_path, 1)  # 1 for fake

# Convert to numpy arrays (only if we have data)
if X and y:
    X = np.array(X)
    y = np.array(y)

    # Scale the features (important for var_smoothing)
    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    # Split into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Define the parameter grid for Gaussian Naive Bayes
    param_grid = {
        'var_smoothing': np.logspace(-10, -7, 10),  # Default is 1e-9
        'priors': [None, [0.5, 0.5], [0.7, 0.3], [0.3, 0.7]]  # Different class priors
    }

    # Initialize the Gaussian Naive Bayes model
    nb = GaussianNB()

    # Define scoring metrics
    scoring = {
        'accuracy': 'accuracy',
        'precision_macro': 'precision_macro',
        'recall_macro': 'recall_macro',
        'f1_macro': 'f1_macro',
        'precision_weighted': 'precision_weighted',
        'recall_weighted': 'recall_weighted',
        'f1_weighted': 'f1_weighted'
    }

    # Set up GridSearchCV
    grid_search = GridSearchCV(
        estimator=nb,
        param_grid=param_grid,
        scoring=scoring,
        cv=5,  # 5-fold cross-validation
        refit='f1_weighted',  # Use weighted F1-score for best model selection
        verbose=1,
        n_jobs=6  # Use all available cores
    )

    # Perform grid search
    grid_search.fit(X_train, y_train)

    # Print the best parameters
    print("Best Parameters:", grid_search.best_params_)
    print("Best Score (Weighted F1):", grid_search.best_score_)

    # Get the best model
    best_nb = grid_search.best_estimator_

    # Make predictions on the test set
    y_pred = best_nb.predict(X_test)

    # Evaluate the model
    print("\nTest Set Performance:")
    print(f"  Accuracy: {accuracy_score(y_test, y_pred):.3f}")
    print(f"  Precision (Macro): {precision_score(y_test, y_pred, average='macro'):.3f}")
    print(f"  Recall (Macro): {recall_score(y_test, y_pred, average='macro'):.3f}")
    print(f"  F1-Score (Macro): {f1_score(y_test, y_pred, average='macro'):.3f}")
    print(f"  Precision (Weighted): {precision_score(y_test, y_pred, average='weighted'):.3f}")
    print(f"  Recall (Weighted): {recall_score(y_test, y_pred, average='weighted'):.3f}")
    print(f"  F1-Score (Weighted): {f1_score(y_test, y_pred, average='weighted'):.3f}")

    # Function to predict a single image
    def predict_image(image_path):
        features = extract_features(image_path)
        if features is not None:
            features = scaler.transform(np.array([features]))
            prediction = best_nb.predict(features)
            proba = best_nb.predict_proba(features)
            return "Fake" if prediction[0] == 1 else "Real", proba[0]
        else:
            return "Error: Unable to process image", None

  

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best Parameters: {'priors': [0.3, 0.7], 'var_smoothing': 1e-10}
Best Score (Weighted F1): 0.6059381505048066

Test Set Performance:
  Accuracy: 0.616
  Precision (Macro): 0.623
  Recall (Macro): 0.617
  F1-Score (Macro): 0.611
  Precision (Weighted): 0.623
  Recall (Weighted): 0.616
  F1-Score (Weighted): 0.611
